In [2]:
import pandas as pd
import numpy as np
import openai

In [3]:
def initialize_openai_client(api_key_file):
    """
    Initialize the OpenAI client with an API key read from a file.

    Args:
    api_key_file (str): The path to the file containing the OpenAI API key.

    Returns:
    openai.Client: An instance of the OpenAI client.
    """
    try:
        with open(api_key_file, 'r') as file:
            api_key = file.read().strip()
        client = openai.Client(api_key=api_key)
        return client
    except Exception as e:
        print(f"An error occurred: {e}")
        return None
    

In [4]:
client = initialize_openai_client("/Users/ahmadrezaie/DalPhD/Research/note_taking_v2/data/input/OpenAIkey.txt")


In [9]:
model = "gpt-4-1106-preview"
temperature=0
max_tokens=4052
top_p=1
frequency_penalty=0
presence_penalty=0


system_prompt = "Assume you are a very experienced physician. You will be given medical notes and your job is to change its format. \nThe notes must be in this format:\n\n1. Subjective: This section includes the patient's own description of their symptoms and complaints.\n\n2. Objective: This section includes observations and data gathered by the physician, such as vital signs, physical examination findings, and test results.\n\n3. Assessment: This section includes the physician's evaluation of the patient's condition, including a diagnosis or differential diagnosis.\n\n4. Plan: This section includes the physician's recommendations for treatment, management, and follow-up.\n You cannot add or remove anything, you must just change the format according to the instruction provided earlier.  "


In [10]:
def change_to_SOAP(note):
        note_response = client.chat.completions.create(
        model=model,
        temperature = temperature,
        max_tokens = max_tokens,
        top_p = top_p,
        frequency_penalty = frequency_penalty,
        presence_penalty = presence_penalty,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": note
            }
        ]
        )
        return note_response.choices[0].message.content


In [14]:
df = pd.read_csv("/Users/ahmadrezaie/papers/Synthetic_Data_Gen/data/input/TaskC-TrainingSet.csv")
df

,dataset,encounter_id,dialogue,note
0,virtassist,D2N001,"[doctor] hi , martha . how are you ?\n[patient...",CHIEF COMPLAINT\n\nAnnual exam.\n\nHISTORY OF ...
1,virtassist,D2N002,"[doctor] hi , andrew , how are you ?\n[patient...",CHIEF COMPLAINT\n\nJoint pain.\n\nHISTORY OF P...
2,virtassist,D2N003,"[doctor] hi , john . how are you ?\n[patient] ...",CHIEF COMPLAINT\n\nBack pain.\n\nHISTORY OF PR...
3,virtassist,D2N004,"[doctor] hi , james , how are you ?\n[patient]...",CHIEF COMPLAINT\n\nBack pain.\n\nHISTORY OF PR...
4,virtassist,D2N005,"[doctor] hey , ms. hill . nice to see you .\n[...",CC:\n\nRight middle finger pain.\n\nHPI:\n\nMs...
...,...,...,...,...
62,aci,D2N063,[doctor] so gloria is a 46 -year-old female to...,CHIEF COMPLAINT\n\nDyspnea.\n\nMEDICAL HISTORY...
63,aci,D2N064,[doctor] hey matthew how're you doing\n[patien...,CHIEF COMPLAINT\n\nLeft ankle pain.\n\nHISTORY...
64,aci,D2N065,[doctor] hey anna good to see you today so i'm...,CHIEF COMPLAINT\n\nRight ankle pain.\n\nHISTOR...
65,aci,D2N066,[doctor] hey gabriel i'm doctor scott good to ...,CHIEF COMPLAINT\n\nBack pain evaluation.\n\nME...


In [15]:
print(df["note"][0])

CHIEF COMPLAINT

Annual exam.

HISTORY OF PRESENT ILLNESS

Martha Collins is a 50-year-old female with a past medical history significant for congestive heart failure, depression, and hypertension who presents for her annual exam. It has been a year since I last saw the patient.

The patient has been traveling a lot recently since things have gotten a bit better. She reports that she got her COVID-19 vaccine so she feels safer about traveling. She has been doing a lot of hiking.

She reports that she is staying active. She has continued watching her diet and she is doing well with that. The patient states that she is avoiding salty foods that she likes to eat. She has continued utilizing her medications. The patient denies any chest pain, shortness of breath, or swelling in her legs.

Regarding her depression, she reports that she has been going to therapy every week for the past year. This has been really helpful for her. She denies suicidal or homicidal ideation.

The patient reports

In [13]:
soap = change_to_SOAP(df["note"][0])
print(soap)

1. Subjective:
Martha Collins, a 50-year-old female with a history of congestive heart failure, depression, and hypertension, presents for her annual exam. She reports staying active with hiking and maintaining a diet low in salt. She has been compliant with her medications and has received her COVID-19 vaccine. She denies experiencing chest pain, shortness of breath, or leg swelling. She attends weekly therapy sessions for depression and denies suicidal or homicidal ideation. However, she admits to occasionally forgetting to take her blood pressure medication, especially during stressful work periods. She also reports nasal congestion due to fall allergies but denies nausea, vomiting, or abdominal pain.

2. Objective:
- Ears, Nose, Mouth, and Throat: Nasal congestion from allergies.
- Cardiovascular: No chest pain or dyspnea on exertion; 3/6 systolic ejection murmur; 1+ pitting edema of the bilateral lower extremities.
- Respiratory: No shortness of breath.
- Gastrointestinal: No abdo

In [16]:
batch_size = 1
processed_count = 0

for index, row in df.iterrows():
    try:
        # Process each row's note to get SOAP content
        df.at[index, 'note_SOAP'] = change_to_SOAP(row['note'])
        processed_count += 1

    except Exception as e:
        # Log the error
        print(f"Error processing row {index}: {e}")
    
    finally:
        # Save the DataFrame every batch_size rows or if an exception occurs
        if processed_count % batch_size == 0 or index == len(df) - 1:
            df.to_csv('/Users/ahmadrezaie/papers/Synthetic_Data_Gen/data/input/TaskC-TrainingSet_SOAP.csv', index=False)
            print(f"Saved progress at row {index}")

# Additional final save, if not already saved
if processed_count % batch_size != 0:
    df.to_csv('/Users/ahmadrezaie/papers/Synthetic_Data_Gen/data/input/TaskC-TrainingSet_SOAP_final.csv', index=False)
    print("All data processed and saved.")

Saved progress at row 0
Saved progress at row 1
Saved progress at row 2
Saved progress at row 3
Saved progress at row 4
Saved progress at row 5
Saved progress at row 6
Saved progress at row 7
Saved progress at row 8
Saved progress at row 9
Saved progress at row 10
Saved progress at row 11
Saved progress at row 12
Saved progress at row 13
Saved progress at row 14
Saved progress at row 15
Saved progress at row 16
Saved progress at row 17
Saved progress at row 18
Saved progress at row 19
Saved progress at row 20
Saved progress at row 21
Saved progress at row 22
Saved progress at row 23
Saved progress at row 24
Saved progress at row 25
Saved progress at row 26
Saved progress at row 27
Saved progress at row 28
Saved progress at row 29
Saved progress at row 30
Saved progress at row 31
Saved progress at row 32
Saved progress at row 33
Saved progress at row 34
Saved progress at row 35
Saved progress at row 36
Saved progress at row 37
Saved progress at row 38
Saved progress at row 39
Saved prog

In [17]:
# test set:

df_test = pd.read_csv("/Users/ahmadrezaie/papers/Synthetic_Data_Gen/data/input/clinicalnlp_taskC_test2.csv")
df_test

,dataset,encounter_id,dialogue,note
0,virtassist,D2N128,"[doctor] hi , carolyn . how are you ?\n[patien...",CHIEF COMPLAINT\n\nFollow-up of chronic proble...
1,virtassist,D2N129,"[doctor] good afternoon , beverly . good to se...",CHIEF COMPLAINT\n\nAnnual exam.\n\nHISTORY OF ...
2,virtassist,D2N130,"[doctor] hi , anna , how are you ?\n[patient] ...",CHIEF COMPLAINT\n\nJoint pain.\n\nHISTORY OF P...
3,virtassist,D2N131,"hi , susan , how are you ?\n[patient] good . h...",CHIEF COMPLAINT\n\nHigh blood pressure check.\...
4,virtassist,D2N132,"[doctor] hello mrs. lee , i see you're here fo...",CC:\n\nBack pain.\n\nHPI:\n\nMs. Lee is a 40-y...
5,virtassist,D2N133,"[doctor] good morning rebecca , nice to see yo...",CHIEF COMPLAINT\n\nJoint pain\n\nHISTORY OF PR...
6,virtassist,D2N134,[doctor] we're gon na go right to the front- ....,CHIEF COMPLAINT\n\nAbnormal labs.\n\nHISTORY O...
7,virtassist,D2N135,"[patient] um , i have high blood sugar . yeah ...",CHIEF COMPLAINT\n\nEvaluation of high blood su...
8,virtassist,D2N136,"[doctor] hi janet , how are you ?\n[patient] g...",CHIEF COMPLAINT\n\nJoint pain.\n\nHISTORY OF P...
9,virtassist,D2N137,"[doctor] morning christine , nice to see you ....",CHIEF COMPLAINT\n\nAnnual exam.\n\nHISTORY OF ...


In [20]:
batch_size = 1
processed_count = 0

for index, row in df_test.iterrows():
    try:
        # Process each row's note to get SOAP content
        df_test.at[index, 'note_SOAP'] = change_to_SOAP(row['note'])
        processed_count += 1

    except Exception as e:
        # Log the error
        print(f"Error processing row {index}: {e}")
    
    finally:
        # Save the DataFrame every batch_size rows or if an exception occurs
        if processed_count % batch_size == 0 or index == len(df) - 1:
            df_test.to_csv('/Users/ahmadrezaie/papers/Synthetic_Data_Gen/data/input/clinicalnlp_taskC_test2_SOAP.csv', index=False)
            print(f"Saved progress at row {index}")

# Additional final save, if not already saved
if processed_count % batch_size != 0:
    df_test.to_csv('/Users/ahmadrezaie/papers/Synthetic_Data_Gen/data/input/clinicalnlp_taskC_test2_SOAP_final.csv', index=False)
    print("All data processed and saved.")

Saved progress at row 0
Saved progress at row 1
Saved progress at row 2
Saved progress at row 3
Saved progress at row 4
Saved progress at row 5
Saved progress at row 6
Saved progress at row 7
Saved progress at row 8
Saved progress at row 9
Saved progress at row 10
Saved progress at row 11
Saved progress at row 12
Saved progress at row 13
Saved progress at row 14
Saved progress at row 15
Saved progress at row 16
Saved progress at row 17
Saved progress at row 18
Saved progress at row 19
Saved progress at row 20
Saved progress at row 21
Saved progress at row 22
Saved progress at row 23
Saved progress at row 24
Saved progress at row 25
Saved progress at row 26
Saved progress at row 27
Saved progress at row 28
Saved progress at row 29
Saved progress at row 30
Saved progress at row 31
Saved progress at row 32
Saved progress at row 33
Saved progress at row 34
Saved progress at row 35
Saved progress at row 36
Saved progress at row 37
Saved progress at row 38
Saved progress at row 39
Error proc